# Gold materialization
Build governed Gold analytics from Silver and record audit validation.

In [ ]:
# ruff: noqa: E501, F821
run_id = "manual"
run_mode = "daily"
bootstrap_signature = ""
scope_kind = "organization"
scope_slug = ""
entity_type = "copilot_usage"
start_date = ""
end_date = ""
lakehouse_files_root = "/lakehouse/default/Files"
bronze_folder = "bronze"
report_types = "entity,users,user-teams,repositories"
silver_schema = "silver"
gold_schema = "gold"
audit_schema = "audit"
telemetry_lag_days = 2


In [ ]:
import json
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

from copilot_metrics_fabric.gold import (
    GOLD_CONTRACTS,
    GoldBuildOptions,
    GoldReplacementWindow,
    normalize_report_types,
)
from copilot_metrics_fabric.gold_spark import build_dataframes, merge_dataframes
from copilot_metrics_fabric.silver import CONTRACTS as SILVER_CONTRACTS
from copilot_metrics_fabric.silver import select_latest_manifests

AUDIT_COLUMNS = ("run_id", "stage", "status", "started_at", "completed_at", "scope_kind", "scope_slug", "start_date", "end_date", "validation_status", "details_json")
def utc_now():
    return datetime.now(timezone.utc).isoformat()
def write_audit(status, started_at, validation_status, details):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{audit_schema}`")
    schema = StructType([StructField(name, StringType(), False) for name in AUDIT_COLUMNS])
    row = (run_id, "gold", status, started_at, utc_now(), scope_kind, scope_slug, start_date, end_date, validation_status, json.dumps(details, sort_keys=True))
    spark.createDataFrame([row], schema).write.format("delta").mode("append").saveAsTable(f"{audit_schema}.pipeline_run_results")


In [ ]:
started_at = utc_now()
try:
    if not scope_slug.strip() or not start_date or not end_date:
        raise ValueError("scope_slug, start_date, and end_date are required")
    replacement_start = date.fromisoformat(start_date)
    replacement_end = date.fromisoformat(end_date)
    input_start = replacement_start - timedelta(days=27)
    expected_report_types = normalize_report_types(report_types)
    excluded = {"silver_quarantine", "silver_schema_drift", "silver_quality_issues"}
    silver_frames = {}
    current_user_frame = None
    for name in SILVER_CONTRACTS:
        qualified = f"{silver_schema}.{name}"
        if name not in excluded and spark.catalog.tableExists(qualified):
            frame = spark.table(qualified)
            if "scope_kind" in frame.columns:
                frame = frame.filter((F.col("scope_kind") == scope_kind) & (F.col("scope_slug") == scope_slug))
            if name == "user_daily":
                current_user_frame = frame
            if "day" in frame.columns:
                frame = frame.filter((F.col("day") >= F.lit(input_start)) & (F.col("day") <= F.lit(replacement_end)))
            silver_frames[name] = frame

    bronze_root = Path(lakehouse_files_root) / bronze_folder
    candidates = []
    for manifest_path in sorted((bronze_root / "_manifests").rglob("*.json")):
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        scope = manifest.get("scope", {})
        report_day = manifest.get("report_day", "")
        if scope.get("kind") == scope_kind and scope.get("slug") == scope_slug and manifest.get("report_type") in expected_report_types and start_date <= report_day <= end_date:
            candidates.append(manifest)
    latest_manifests = select_latest_manifests(candidates)
    report_status_rows = [{
        "scope_kind": manifest["scope"]["kind"],
        "scope_slug": manifest["scope"]["slug"],
        "day": manifest["report_day"],
        "report_type": manifest["report_type"],
        "status": "complete" if manifest["status"] == "success" else "no_data",
        "ingestion_id": manifest.get("ingestion_id"),
        "source_ingested_at": manifest.get("ingested_at"),
    } for manifest in latest_manifests]
    if not silver_frames and not report_status_rows:
        raise RuntimeError("no Silver tables or Bronze manifests were available for the requested scope and date range")
    if current_user_frame is None:
        raise RuntimeError("silver.user_daily is required to recompute current user state from complete history")
    options = GoldBuildOptions(as_of_day=replacement_end, calendar_start=replacement_start, calendar_end=replacement_end, telemetry_lag_days=int(telemetry_lag_days), expected_report_types=expected_report_types)
    frames = build_dataframes(spark, silver_frames, options=options, report_status_rows=report_status_rows, current_user_frame=current_user_frame)
    windows = [GoldReplacementWindow(scope_kind, scope_slug, start_date, end_date)]
    merge_dataframes(spark, frames, schema=gold_schema, replacement_windows=windows)
    missing = sorted(set(GOLD_CONTRACTS) - set(frames))
    if missing:
        raise RuntimeError(f"Gold contract outputs missing: {missing}")
    details = {"entity_type": entity_type, "run_mode": run_mode, "bootstrap_signature": bootstrap_signature, "silver_tables": sorted(silver_frames), "latest_manifests": len(latest_manifests), "table_rows": {name: frame.count() for name, frame in frames.items()}}
    write_audit("succeeded", started_at, "passed", details)
except Exception as error:
    try:
        write_audit("failed", started_at, "failed", {"entity_type": entity_type, "error_type": type(error).__name__})
    finally:
        raise
